In [ ]:
# Hybrid "Content-based" - похожие товары по характеристикам, лог. регрессия

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
import hdbscan
from sklearn.manifold import TSNE
import umap
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
import openpyxl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.compose import ColumnTransformer
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [2]:
#df - транзакционный очищенный датасет + кластеры каждого пользователя
#client_df - агрегированный датасет по пользователям
#df_test - тестовый датасет
df = pd.read_parquet('df.parquet', engine='fastparquet')
client_df = pd.read_parquet('client_data.parquet', engine='fastparquet')
df_test = pd.read_parquet('df_test.parquet', engine='fastparquet')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 695703 entries, 0 to 695702
Data columns (total 39 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   Дата                      695703 non-null  datetime64[ns]
 1   ДатаДоставки              695703 non-null  datetime64[ns]
 2   НомерЗаказаНаСайте        695703 non-null  object        
 3   НовыйСтатус               695703 non-null  category      
 4   СуммаЗаказаНаСайте        695703 non-null  float64       
 5   СуммаДокумента            695703 non-null  float64       
 6   МетодДоставки             695703 non-null  category      
 7   ФормаОплаты               695703 non-null  category      
 8   Регион                    693088 non-null  category      
 9   Группа2                   695703 non-null  category      
 10  Группа3                   695703 non-null  category      
 11  Группа4                   664259 non-null  category      
 12  Ти

In [4]:
client_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80795 entries, 0 to 80794
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Телефон_new        80795 non-null  object 
 1   orders_count       80795 non-null  float64
 2   items_total        80795 non-null  float64
 3   revenue_total      80795 non-null  float64
 4   avg_price          80795 non-null  float64
 5   margin_total       80795 non-null  float64
 6   unique_categories  80795 non-null  int64  
 7   recency_days       80795 non-null  int64  
 8   lifetime_days      80795 non-null  int64  
 9   avg_check          80795 non-null  float64
 10  items_per_order    80795 non-null  float64
 11  cluster_5          80795 non-null  int32  
dtypes: float64(7), int32(1), int64(3), object(1)
memory usage: 7.1+ MB


In [29]:
#Ограничение выборки для тестирования
TEST_MODE = False
SAMPLE_USERS = 500
CANDIDATES_PER_USER = 500

# Полный справочник товаров
df_full = pd.concat([pd.read_parquet('df.parquet', engine='fastparquet'),
                     pd.read_parquet('df_test.parquet', engine='fastparquet')], 
                    ignore_index=True)

item_features = df_full[['ID_SKU', 'Группа2', 'Группа3', 'ТипТовара', 'Цена']].drop_duplicates('ID_SKU')
for col in ['Группа2', 'Группа3', 'ТипТовара']:
    item_features[col] = item_features[col].astype('object').fillna('unknown')

all_items = item_features['ID_SKU'].unique()
TOP_ITEMS = 5000
popular_items = df_full['ID_SKU'].value_counts().head(TOP_ITEMS).index.tolist()

print(f"Всего товаров в справочнике: {len(all_items)}")

# Данные для обучения и теста
df = pd.read_parquet('df.parquet', engine='fastparquet')
client_df = pd.read_parquet('client_data.parquet', engine='fastparquet')
df_test = pd.read_parquet('df_test.parquet', engine='fastparquet')
df_test = df_test.merge(client_df[['Телефон_new', 'cluster_5']], on='Телефон_new', how='inner')

if TEST_MODE:
    sample_users = np.random.choice(df['Телефон_new'].unique(), SAMPLE_USERS, replace=False)
    df = df[df['Телефон_new'].isin(sample_users)]
    client_df = client_df[client_df['Телефон_new'].isin(sample_users)]
    df_test = df_test[df_test['Телефон_new'].isin(sample_users)]
    print(f"[ТЕСТОВЫЙ РЕЖИМ] Пользователей: {df['Телефон_new'].nunique()}, строк: {len(df)}")
else:
    print(f"[ПОЛНЫЙ ПРОГОН] Пользователей: {df['Телефон_new'].nunique()}, строк: {len(df)}")

print(f"Кластеры: {sorted(df['cluster_5'].unique())}")

Всего товаров в справочнике: 138046
[ПОЛНЫЙ ПРОГОН] Пользователей: 80795, строк: 695703
Кластеры: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4)]


In [30]:
# Характеристики товаров и популярные товары
print(f"Уникальных товаров в справочнике: {len(all_items)}")
print(f"Популярных товаров: {len(popular_items)}")

Уникальных товаров в справочнике: 138046
Популярных товаров: 5000


In [31]:
# Пользовательские признаки

user_features = client_df[['Телефон_new', 'orders_count', 'avg_check', 'recency_days', 
                           'lifetime_days', 'unique_categories']].copy()
user_avg_price = df.groupby('Телефон_new')['Цена'].mean().reset_index(name='user_avg_price')



In [32]:
# Функция создания обучающей выборки для одного кластера
def build_train_data(df_cluster, all_items):
    positive_df = df_cluster[['Телефон_new', 'ID_SKU', 'cluster_5']].copy()
    positive_df['label'] = 1
    
    negative_samples = []
    users_unique = positive_df['Телефон_new'].unique()
    
    for user in users_unique:
        bought = set(df_cluster[df_cluster['Телефон_new'] == user]['ID_SKU'].unique())
        available = [item for item in all_items if item not in bought]
        
        user_pos = positive_df[positive_df['Телефон_new'] == user]
        n_neg = len(user_pos) * 5
        
        if len(available) >= n_neg:
            neg_items = np.random.choice(available, n_neg, replace=False)
        else:
            neg_items = np.random.choice(available, n_neg, replace=True)
        
        cluster = user_pos['cluster_5'].iloc[0]
        for item in neg_items:
            negative_samples.append([user, item, cluster, 0])
    
    negative_df = pd.DataFrame(negative_samples, columns=['Телефон_new', 'ID_SKU', 'cluster_5', 'label'])
    return pd.concat([positive_df, negative_df], ignore_index=True)

In [33]:
# Функция добавления признаков
def add_features(train_ml, item_features, user_features, user_avg_price, df_cluster):
    train_ml = train_ml.merge(item_features, on='ID_SKU', how='left')
    train_ml = train_ml.merge(user_features, on='Телефон_new', how='left')
    
    user_item_count = df_cluster.groupby(['Телефон_new', 'ID_SKU']).size().reset_index(name='user_item_count')
    train_ml = train_ml.merge(user_item_count, on=['Телефон_new', 'ID_SKU'], how='left')
    train_ml['user_item_count'] = train_ml['user_item_count'].fillna(0)
    
    train_ml = train_ml.merge(user_avg_price, on='Телефон_new', how='left')
    train_ml['price_diff'] = train_ml['Цена'] - train_ml['user_avg_price']
    train_ml['price_diff'] = train_ml['price_diff'].fillna(0)
    
    train_ml['Цена'] = train_ml['Цена'].fillna(train_ml['Цена'].median())
    for col in ['Группа2', 'Группа3', 'ТипТовара']:
        train_ml[col] = train_ml[col].fillna('unknown')
    
    return train_ml

In [34]:
# Обучение моделей для каждого кластера
cat_cols = ['Группа2', 'Группа3', 'ТипТовара']
numeric_cols = ['Цена', 'orders_count', 'avg_check', 'recency_days', 'lifetime_days', 
                'unique_categories', 'price_diff']

models = {}
preprocessors = {}
results_val = {}

clusters = sorted(df['cluster_5'].unique())

for cluster_id in tqdm(clusters, desc="Кластеры"):
    print(f"\nКластер {cluster_id}: пользователей {df[df['cluster_5']==cluster_id]['Телефон_new'].nunique()}")
    
    df_cluster = df[df['cluster_5'] == cluster_id]
    
    print("  Сборка данных", end=' ')
    train_ml = build_train_data(df_cluster, all_items)
    train_ml = add_features(train_ml, item_features, user_features, user_avg_price, df_cluster)
    print(f"→ {len(train_ml)} примеров")
    
    all_users = train_ml['Телефон_new'].unique()
    train_users, val_users = train_test_split(all_users, test_size=0.1, random_state=42)
    
    X_train = train_ml[train_ml['Телефон_new'].isin(train_users)]
    X_val = train_ml[train_ml['Телефон_new'].isin(val_users)]
    
    print("  Кодирование", end=' ')
    preprocessor = ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', StandardScaler(), numeric_cols)
    ])
    X_train_enc = preprocessor.fit_transform(X_train)
    y_train = X_train['label'].values
    X_val_enc = preprocessor.transform(X_val)
    y_val = X_val['label'].values
    print(f"→ {X_train_enc.shape[1]} признаков")
    
    best_ap = -1
    best_params = {}
    
    for C in tqdm([0.01, 0.1, 1.0, 10.0], desc="  Подбор C", leave=False):
        for cw in [None, 'balanced']:
            model = LogisticRegression(C=C, class_weight=cw, max_iter=1000, random_state=42)
            model.fit(X_train_enc, y_train)
            ap = average_precision_score(y_val, model.predict_proba(X_val_enc)[:, 1])
            if ap > best_ap:
                best_ap = ap
                best_params = {'C': C, 'class_weight': cw}
    
    print(f"  Лучшие: {best_params}, AP={best_ap:.4f}")
    
    print("  Финальная модель", end=' ')
    X_all = preprocessor.fit_transform(train_ml)
    y_all = train_ml['label'].values
    final_model = LogisticRegression(**best_params, max_iter=1000, random_state=42)
    final_model.fit(X_all, y_all)
    print("✓")
    
    models[cluster_id] = final_model
    preprocessors[cluster_id] = preprocessor
    results_val[cluster_id] = best_ap

print(f"\nГотово: " + ", ".join([f"кластер {c}: AP={ap:.4f}" for c, ap in results_val.items()]))


Кластеры:   0%|                                           | 0/5 [00:00<?, ?it/s]


Кластер 0: пользователей 17368
  Сборка данных → 233496 примеров
  Кодирование → 120 признаков



  Подбор C: 100%|█████████████████████████████████| 4/4 [00:05<00:00,  1.51s/it]
                                                                                

  Лучшие: {'C': 10.0, 'class_weight': None}, AP=0.4012
  Финальная модель 

Кластеры:  20%|██████▊                           | 1/5 [04:26<17:44, 266.01s/it]

...

✓

Кластер 4: пользователей 25508
  Сборка данных → 335046 примеров
  Кодирование → 120 признаков



  Подбор C: 100%|█████████████████████████████████| 4/4 [00:08<00:00,  2.31s/it]
                                                                                

  Лучшие: {'C': 1.0, 'class_weight': 'balanced'}, AP=0.5040
  Финальная модель 

Кластеры: 100%|██████████████████████████████████| 5/5 [28:23<00:00, 340.64s/it]

✓

Готово: кластер 0: AP=0.4012, кластер 1: AP=0.4748, кластер 2: AP=0.4639, кластер 3: AP=0.5763, кластер 4: AP=0.5040


In [35]:
# Функция рекомендаций
def recommend_hybrid(user_id, n=10):
    if user_id not in client_df['Телефон_new'].values:
        return popular_items[:n]
    
    user_row = client_df[client_df['Телефон_new'] == user_id].iloc[0]
    cluster = user_row['cluster_5']
    
    if cluster not in models:
        return popular_items[:n]
    
    user_data = {
        'orders_count': user_row.get('orders_count', 0),
        'avg_check': user_row.get('avg_check', 0),
        'recency_days': user_row.get('recency_days', 365),
        'lifetime_days': user_row.get('lifetime_days', 0),
        'unique_categories': user_row.get('unique_categories', 0)
    }
    
    uap = user_avg_price[user_avg_price['Телефон_new'] == user_id]['user_avg_price']
    user_avg_price_val = uap.iloc[0] if len(uap) > 0 else user_data['avg_check']
    
    bought = set(df[df['Телефон_new'] == user_id]['ID_SKU'].unique())
    candidates = list(set(all_items) - bought)
    
    rng = np.random.RandomState(hash(user_id) % 2**31)
    rng.shuffle(candidates)
    candidates = candidates[:CANDIDATES_PER_USER]
    
    if len(candidates) == 0:
        return popular_items[:n]
    
    rows = []
    for item in candidates:
        ir = item_features[item_features['ID_SKU'] == item]
        if ir.empty:
            continue
        ir = ir.iloc[0]
        rows.append({
            'ID_SKU': item,
            'Группа2': str(ir['Группа2']),
            'Группа3': str(ir['Группа3']),
            'ТипТовара': str(ir['ТипТовара']),
            'Цена': ir['Цена'],
            'orders_count': user_data['orders_count'],
            'avg_check': user_data['avg_check'],
            'recency_days': user_data['recency_days'],
            'lifetime_days': user_data['lifetime_days'],
            'unique_categories': user_data['unique_categories'],
            'user_item_count': 0,
            'price_diff': ir['Цена'] - user_avg_price_val
        })
    
    candidates_df = pd.DataFrame(rows)
    X_cand = preprocessors[cluster].transform(candidates_df)
    candidates_df['score'] = models[cluster].predict_proba(X_cand)[:, 1]
    
    return candidates_df.sort_values('score', ascending=False)['ID_SKU'].head(n).tolist()

In [36]:
# Группировка теста по пользователям
test_grouped = df_test.groupby('Телефон_new').agg(
    true_items=('ID_SKU', list),
    cluster=('cluster_5', 'first')
).reset_index()

print(f"Пользователей в тесте: {len(test_grouped)}")
print(f"Среднее товаров в заказе: {test_grouped['true_items'].apply(len).mean():.1f}")

Пользователей в тесте: 80795
Среднее товаров в заказе: 3.2


In [37]:
# Диагностика одного пользователя
user = test_grouped['Телефон_new'].iloc[0]
true_items = test_grouped[test_grouped['Телефон_new'] == user]['true_items'].iloc[0]
cluster = test_grouped[test_grouped['Телефон_new'] == user]['cluster'].iloc[0]

recs = recommend_hybrid(user, n=10)
bought = set(df[df['Телефон_new'] == user]['ID_SKU'].unique())
candidates = list(set(all_items) - bought)[:CANDIDATES_PER_USER]

print(f"User: {user}")
print(f"Cluster: {cluster}")
print(f"True items: {true_items}")
print(f"Recommendations: {recs}")
print(f"True items in all_items: {[i for i in true_items if i in all_items]}")
print(f"True items in candidates: {[i for i in true_items if i in candidates]}")

User: 55555748-48484848484870
Cluster: 0
True items: ['IDL00003532755']
Recommendations: ['ID9010018016856', 'IDL00039254048', 'ID000sm-0595452', 'ID10005589351', 'IDL00030348351', 'IDL00028398149', 'IDL00028723856', 'IDL00040969048', 'ID10013477755', 'ID10011713654']
True items in all_items: ['IDL00003532755']
True items in candidates: []


In [38]:
user = '55574848-49575652514879'
bought = set(df[df['Телефон_new'] == user]['ID_SKU'].unique())
true_items = ['ID55096048', 'ID10002538654', 'IDL00052433755', 'IDL00035789149', 'IDL00045643957']

print(f"bought size: {len(bought)}")
print(f"True items in bought: {[i for i in true_items if i in bought]}")
print(f"all_items size: {len(all_items)}")
print(f"all_items - bought size: {len(set(all_items) - bought)}")

# Проверяем, есть ли true_items вообще в all_items
for item in true_items:
    in_all = item in all_items
    in_bought = item in bought
    print(f"  {item}: in_all={in_all}, in_bought={in_bought}")

bought size: 6
True items in bought: []
all_items size: 138046
all_items - bought size: 138040
  ID55096048: in_all=True, in_bought=False
  ID10002538654: in_all=True, in_bought=False
  IDL00052433755: in_all=True, in_bought=False
  IDL00035789149: in_all=True, in_bought=False
  IDL00045643957: in_all=True, in_bought=False


In [39]:
# Функция оценки на уровне пользователей
def evaluate_model_user_level(test_grouped, recommender_func, k=10):
    hits = 0
    map_sum = 0.0
    
    for _, row in test_grouped.iterrows():
        user = row['Телефон_new']
        true_items = row['true_items']
        
        try:
            recs = recommender_func(user, k)
        except:
            continue
        
        hits_in_recs = [item for item in true_items if item in recs]
        if len(hits_in_recs) > 0:
            hits += 1
            map_sum += np.mean([1.0 / (recs.index(item) + 1) for item in hits_in_recs])
    
    return {'HitRate@K': hits / len(test_grouped), 'MAP@K': map_sum / len(test_grouped)}

In [51]:
# Оценка на полном тесте
for k in [5, 10, 20]:
    m = evaluate_model_user_level(test_grouped, recommend_hybrid, k=k)
    print(f"K={k}: HR = {m['HitRate@K']:.4f}, MAP = {m['MAP@K']:.4f}")

K=5: HR = 0.0750, MAP = 0.0330
K=10: HR = 0.1180, MAP = 0.0460
K=20: HR = 0.1690, MAP = 0.0580


In [52]:
# Оценка по кластерам
for c in sorted(test_grouped['cluster'].unique()):
    ct = test_grouped[test_grouped['cluster'] == c]
    if len(ct) == 0:
        continue
    m = evaluate_model_user_level(ct, recommend_hybrid, k=10)
    print(f"Кластер {c}: n={len(ct)}, HR@10={m['HitRate@K']:.4f}, MAP@10={m['MAP@K']:.4f}")

Кластер 0: n=17368, HR@10=0.0510, MAP@10=0.0190
Кластер 1: n=12768, HR@10=0.1610, MAP@10=0.0610
Кластер 2: n=19387, HR@10=0.1330, MAP@10=0.0490
Кластер 3: n=5764, HR@10=0.0940, MAP@10=0.0330
Кластер 4: n=25508, HR@10=0.0670, MAP@10=0.0210


In [ ]:
# Сохранение результатов
results_hybrid = test_grouped[['Телефон_new', 'cluster']].copy()
results_hybrid.columns = ['Телефон_new', 'cluster_5']

hits_list = []
maps_list = []

for _, row in tqdm(test_grouped.iterrows(), total=len(test_grouped), desc="Сохранение"):
    user = row['Телефон_new']
    true_items = row['true_items']
    
    recs = recommend_hybrid(user, n=10)
    
    hits_in_recs = [item for item in true_items if item in recs]
    if len(hits_in_recs) > 0:
        hits_list.append(1)
        maps_list.append(np.mean([1.0 / (recs.index(item) + 1) for item in hits_in_recs]))
    else:
        hits_list.append(0)
        maps_list.append(0.0)

results_hybrid['hybrid_hit'] = hits_list
results_hybrid['hybrid_map'] = maps_list
results_hybrid.to_parquet('results_hybrid.parquet', engine='fastparquet', index=False)